In [1]:
from ultralytics import YOLO

import torch
import torchvision.transforms as transforms

from torchvision.models import resnet18
import torchvision.models as models
from PIL import Image

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
import shutil
from tqdm import tqdm
import time

In [2]:
YOLO_MODEL = YOLO("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\runs\\detect\\runs\\YOLOv8_baseline\\weights\\best.pt")

In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cnn = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = cnn.fc.in_features
cnn.fc = torch.nn.Sequential(
    torch.nn.Linear(num_features, 128),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.5),
    torch.nn.Linear(128, 2)
)

cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\notebooks\\best_chicken_cnn.pth"))

cnn.to(DEVICE)

cnn.eval()

C:\Users\klanz\AppData\Local\Temp\ipykernel_10364\2231296117.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn.load_state_dict(torch.load("C:\\Users\\klanz\\Desktop\\

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [4]:
transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )

])

In [5]:
def classify_crop(crop):

    image = Image.fromarray(cv2.cvtColor(crop,cv2.COLOR_BGR2RGB))

    tensor = transform(image)

    tensor = tensor.unsqueeze(0)

    tensor = tensor.to(DEVICE)

    with torch.no_grad():

        prediction = cnn(tensor)

        prediction = prediction.argmax(1).item()

    return prediction

In [6]:
def door_decision(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    cnn_classes = [p["cnn_prediction"] for p in predictions]

    if 1 in cnn_classes:
        return "CLOSE"

    if 0 in cnn_classes:
        return "OPEN"

    return "CLOSE"

In [7]:
def door_decision_yolo(predictions):

    if len(predictions) == 0:
        return "CLOSE"

    classes = [p["class"] for p in predictions]

    if 1 in classes:
        return "CLOSE"

    if 0 in classes:
        return "OPEN"

    return "CLOSE"

In [8]:
CLASS_NAMES = {
    0: "chicken",
    1: "not_chicken"
}

In [9]:
def run_yolo(image_path, conf=0.25):

    result = YOLO_MODEL.predict(
        source=str(image_path),
        conf=conf,
        verbose=False
    )[0]

    predictions = []

    for box in result.boxes:

        cls = int(box.cls.item())

        confidence = float(box.conf.item())

        x1, y1, x2, y2 = box.xyxy.cpu().numpy()[0]

        predictions.append({

            "class": cls,
            "class_name": CLASS_NAMES[cls],
            "confidence": confidence,
            "bbox": [int(x1), int(y1), int(x2), int(y2)]

        })

    return predictions

In [10]:
def run_pipeline(image_path, conf=0.25):

    detections = run_yolo(image_path, conf)

    image = cv2.imread(str(image_path))

    final_predictions = []

    for det in detections:

        x1, y1, x2, y2 = det["bbox"]

        crop = image[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        cnn_prediction = classify_crop(crop)

        final_predictions.append({

            "yolo_prediction": det["class"],

            "cnn_prediction": cnn_prediction,

            "confidence": det["confidence"],

            "bbox": det["bbox"]

        })

    return final_predictions

In [11]:
def load_ground_truth(label_path):

    if not Path(label_path).exists():
        return []

    gt=[]

    with open(label_path) as f:

        for line in f:

            line=line.strip()

            if line=="":

                continue

            cls=int(line.split()[0])

            gt.append(cls)

    return gt

In [12]:
def ground_truth_decision(gt_classes):

    if 1 in gt_classes:
        return "CLOSE"

    if 0 in gt_classes:
        return "OPEN"

    return "CLOSE"

In [13]:
test_images = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset\\test\\images").glob("*"))

results = []

In [14]:
for image_path in test_images:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")

    gt = load_ground_truth(label_path)

    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()

    yolo_predictions = run_yolo(image_path)

    yolo_time = (time.perf_counter()-start)*1000

    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()

    pipeline_predictions = run_pipeline(image_path)

    pipeline_time = (time.perf_counter()-start)*1000

    pipeline_decision = door_decision(pipeline_predictions)

    results.append({

        "image": image_path.name,

        "ground_truth": gt_decision,

        "yolo": yolo_decision,

        "pipeline": pipeline_decision,

        "yolo_time_ms": yolo_time,

        "pipeline_time_ms": pipeline_time,

        "objects_gt": gt,

        "objects_yolo": [x["class"] for x in yolo_predictions],

        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [15]:
df = pd.DataFrame(results)

df.head(20)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,516.7432,98.1776,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,22.9747,19.6432,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,15.2921,23.5043,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,15.2042,20.5770,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,14.1893,38.4920,"[0, 0, 0]","[0, 0, 0, 0, 0]","[0, 0, 0, 1, 1]"
5,1085.jpeg,OPEN,OPEN,CLOSE,14.0078,16.7186,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,27.9492,38.1366,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,15.8802,25.6250,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,16.9037,19.7068,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,21.7339,36.4427,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"


In [16]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix

import numpy as np

In [17]:
decision_map = {

    "OPEN":1,

    "CLOSE":0

}

In [18]:
def evaluate_system(df, prediction_column, time_column):

    gt = df["ground_truth"].map(decision_map)

    pred = df[prediction_column].map(decision_map)

    accuracy = accuracy_score(gt, pred)

    precision = precision_score(
        gt,
        pred,
        zero_division=0
    )

    recall = recall_score(
        gt,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        gt,
        pred,
        zero_division=0
    )

    cm = confusion_matrix(gt, pred)

    tn, fp, fn, tp = cm.ravel()

    avg_time = df[time_column].mean()

    print("="*50)

    print(prediction_column)

    print("="*50)

    print(f"Accuracy : {accuracy:.4f}")

    print(f"Precision: {precision:.4f}")

    print(f"Recall   : {recall:.4f}")

    print(f"F1-score : {f1:.4f}")

    print()

    print(f"TP : {tp}")

    print(f"FP : {fp}")

    print(f"TN : {tn}")

    print(f"FN : {fn}")

    print()

    print(f"Średni czas: {avg_time:.2f} ms")

    return {

        "Accuracy":accuracy,

        "Precision":precision,

        "Recall":recall,

        "F1":f1,

        "TP":tp,

        "FP":fp,

        "TN":tn,

        "FN":fn,

        "Time":avg_time

    }

In [19]:
yolo_results = evaluate_system(

    df,

    "yolo",

    "yolo_time_ms"

)

yolo
Accuracy : 0.9943
Precision: 0.9959
Recall   : 0.9938
F1-score : 0.9949

TP : 484
FP : 2
TN : 389
FN : 3

Średni czas: 17.10 ms


In [20]:
pipeline_results = evaluate_system(

    df,

    "pipeline",

    "pipeline_time_ms"

)

pipeline
Accuracy : 0.9875
Precision: 0.9938
Recall   : 0.9836
F1-score : 0.9886

TP : 479
FP : 3
TN : 388
FN : 8

Średni czas: 26.21 ms


In [21]:
comparison = pd.DataFrame(

    [

        yolo_results,

        pipeline_results

    ],

    index=[

        "YOLO",

        "YOLO + CNN"

    ]

)

comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.994305,0.995885,0.993840,0.994861,484,2,389,3,17.104392
YOLO + CNN,0.987472,0.993776,0.983573,0.988648,479,3,388,8,26.214541


In [22]:
dangerous_yolo = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN")

]

In [23]:
dangerous_pipeline = df[

    (df["ground_truth"]=="CLOSE") &

    (df["pipeline"]=="OPEN")

]

In [24]:
print()

print("Krytyczne błędy")

print("----------------")

print("YOLO:",len(dangerous_yolo))

print("YOLO+CNN:",len(dangerous_pipeline))


Krytyczne błędy
----------------
YOLO: 2
YOLO+CNN: 3


In [25]:
improved = df[

    (df["ground_truth"]=="CLOSE") &

    (df["yolo"]=="OPEN") &

    (df["pipeline"]=="CLOSE")

]

improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
333,Image-84-bd2f1b.jpg,CLOSE,OPEN,CLOSE,13.6166,21.559,[1],"[0, 0]","[1, 1]"


In [26]:
empty_images = 0
false_detections = 0

for row in results:

    if len(row["objects_gt"]) == 0:

        empty_images += 1

        if len(row["objects_yolo"]) > 0:
            false_detections += 1

print("Puste obrazy:",empty_images)
print("Fałszywe detekcje:",false_detections)

if empty_images>0:

    print(
        "Odsetek:",
        false_detections/empty_images
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [27]:
fp_images=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["pipeline"]=="OPEN":

        fp_images.append(row["image"])

print("False Positive:",len(fp_images))

fp_images

False Positive: 3


['Image-51-eb7770.jpg',
 'raptor__gbif_raptor_00254_jpg.rf.RUznLns9phCl4pmygjB7.jpg',
 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg']

In [28]:
fn_images=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["pipeline"]=="CLOSE":

        fn_images.append(row["image"])

print("False Negative:",len(fn_images))

fn_images

False Negative: 8


['1054.jpeg',
 '1085.jpeg',
 '561.jpeg',
 'neg_poultry__poultry_244_jpg.rf.X1DkC0GjdbMdNEwCaauz.jpg',
 'OIP-9Q37DlviKYpHMGPjddECLgAAAA.jpeg',
 'OIP-FbBPwSbJW6mmwK-7AlqNRAHaEK.jpeg',
 'OIP-K4mN6ZEHdRTScC-_uO6G7wHaFi.jpeg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg']

In [29]:
fp_yolo=[]

for row in results:

    if row["ground_truth"]=="CLOSE" and row["yolo"]=="OPEN":

        fp_yolo.append(row["image"])

len(fp_yolo)

fp_yolo

['Image-84-bd2f1b.jpg', 'raptor__raptor_028_jpg.rf.a9tIjTTL30oWJULOxUVa.jpg']

In [30]:
fn_yolo=[]

for row in results:

    if row["ground_truth"]=="OPEN" and row["yolo"]=="CLOSE":

        fn_yolo.append(row["image"])

len(fn_yolo)

fn_yolo

['neg_poultry__poultry_222_jpg.rf.g0AgDML3T5PV9uIHp0Xd.jpg',
 'neg_poultry__poultry_244_jpg.rf.X1DkC0GjdbMdNEwCaauz.jpg',
 'OIP-R1UhJyfXM5EoZdTAA2WzygHaFj.jpeg']

In [31]:
def darker(image, factor=0.5):

    img = image.astype(np.float32)
    img *= factor
    img = np.clip(img,0,255)

    return img.astype(np.uint8)


def night(image):

    img = darker(image,0.30)

    blue = np.zeros_like(img)
    blue[:,:,0]=30

    img = cv2.addWeighted(img,1.0,blue,0.25,0)

    return img


def random_occlusion(image):

    img=image.copy()

    h,w=img.shape[:2]

    occ_w=np.random.randint(w//8,w//2)
    occ_h=np.random.randint(h//8,h//2)

    x=np.random.randint(0,w-occ_w)
    y=np.random.randint(0,h-occ_h)

    cv2.rectangle(
        img,
        (x,y),
        (x+occ_w,y+occ_h),
        (0,0,0),
        -1
    )

    return img

def motion_blur(image, kernel_size=15, angle=0):

    if kernel_size % 2 == 0:
        kernel_size += 1

    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)

    kernel[kernel_size // 2, :] = 1.0

    center = (kernel_size / 2 - 0.5, kernel_size / 2 - 0.5)

    rotation_matrix = cv2.getRotationMatrix2D(
        center,
        angle,
        1.0
    )

    kernel = cv2.warpAffine(
        kernel,
        rotation_matrix,
        (kernel_size, kernel_size)
    )

    kernel_sum = kernel.sum()

    if kernel_sum != 0:
        kernel /= kernel_sum

    blurred = cv2.filter2D(
        image,
        -1,
        kernel
    )

    return blurred

In [32]:
TEST_FOLDER = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test")

OUTPUT = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset_experiments")

conditions = [
    "normal",
    "dark",
    "night",
    "occlusion",
    "motion_blur"
]

In [33]:
for cond in conditions:

    (OUTPUT/cond/"images").mkdir(parents=True,exist_ok=True)
    (OUTPUT/cond/"labels").mkdir(parents=True,exist_ok=True)
    

In [34]:
image_folder = TEST_FOLDER/"images"
label_folder = TEST_FOLDER/"labels"

images=list(image_folder.glob("*"))

In [35]:
for image_path in tqdm(images):

    image=cv2.imread(str(image_path))

    label=label_folder/(image_path.stem+".txt")

    shutil.copy(label,OUTPUT/"normal"/"labels"/label.name)
    shutil.copy(label,OUTPUT/"dark"/"labels"/label.name)
    shutil.copy(label,OUTPUT/"night"/"labels"/label.name)
    shutil.copy(label,OUTPUT/"occlusion"/"labels"/label.name)
    shutil.copy(label,OUTPUT/"motion_blur"/"labels"/label.name)

    cv2.imwrite(
        str(OUTPUT/"normal"/"images"/image_path.name),
        image
    )

    cv2.imwrite(
        str(OUTPUT/"dark"/"images"/image_path.name),
        darker(image,0.45)
    )

    cv2.imwrite(
        str(OUTPUT/"night"/"images"/image_path.name),
        night(image)
    )

    cv2.imwrite(
        str(OUTPUT/"occlusion"/"images"/image_path.name),
        random_occlusion(image)
    )

    cv2.imwrite(
    str(OUTPUT/"motion_blur"/"images"/image_path.name),
    motion_blur(image, kernel_size=15, angle=np.random.randint(0, 180))
    )

100%|██████████| 878/878 [00:35<00:00, 25.04it/s]


In [36]:
"""for image_path in tqdm(images):

    image=cv2.imread(str(image_path))

    label=label_folder/(image_path.stem+".txt")

    shutil.copy(label,OUTPUT/"occlusion"/"labels"/label.name)


    cv2.imwrite(
        str(OUTPUT/"occlusion"/"images"/image_path.name),
        random_occlusion(image)
    )"""

'for image_path in tqdm(images):\n\n    image=cv2.imread(str(image_path))\n\n    label=label_folder/(image_path.stem+".txt")\n\n    shutil.copy(label,OUTPUT/"occlusion"/"labels"/label.name)\n\n\n    cv2.imwrite(\n        str(OUTPUT/"occlusion"/"images"/image_path.name),\n        random_occlusion(image)\n    )'

In [37]:
image_path_dark = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\dark\\images").glob("*"))
image_path_night = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\night\\images").glob("*"))
image_path_occlusion = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\occlusion\\images").glob("*"))
image_path_motion_blur = sorted(Path("C:\\Users\\klanz\\Desktop\\MAGISTERKA\\dataset_experiments\\motion_blur\\images").glob("*"))

results_dark = []
results_night = []
results_occlusion = []
results_motion_blur = []


In [38]:
for image_path in image_path_dark:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_dark.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [39]:
for image_path in image_path_night:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_night.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [40]:
for image_path in image_path_occlusion:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_occlusion.append({
        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [41]:
for image_path in image_path_motion_blur:

    label_path = Path(r"C:\Users\klanz\Desktop\MAGISTERKA\dataset\test\labels") / (image_path.stem + ".txt")
    gt = load_ground_truth(label_path)
    gt_decision = ground_truth_decision(gt)

    # ---------- YOLO ----------

    start = time.perf_counter()
    yolo_predictions = run_yolo(image_path)
    yolo_time = (time.perf_counter()-start)*1000
    yolo_decision = door_decision_yolo(yolo_predictions)

    # ---------- PIPELINE ----------

    start = time.perf_counter()
    pipeline_predictions = run_pipeline(image_path)
    pipeline_time = (time.perf_counter()-start)*1000
    pipeline_decision = door_decision(pipeline_predictions)

    results_motion_blur.append({

        "image": image_path.name,
        "ground_truth": gt_decision,
        "yolo": yolo_decision,
        "pipeline": pipeline_decision,
        "yolo_time_ms": yolo_time,
        "pipeline_time_ms": pipeline_time,
        "objects_gt": gt,
        "objects_yolo": [x["class"] for x in yolo_predictions],
        "objects_pipeline": [x["cnn_prediction"] for x in pipeline_predictions]

    })

In [42]:
df_dark = pd.DataFrame(results_dark)
df_night = pd.DataFrame(results_night)
df_occlusion = pd.DataFrame(results_occlusion)
df_motion_blur = pd.DataFrame(results_motion_blur)

df_dark.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,82.0014,81.3389,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,67.2575,17.2694,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,29.1774,19.0569,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,26.9266,21.8894,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,31.4713,31.3143,"[0, 0, 0]","[0, 0, 0, 0, 0]","[0, 0, 0, 0, 1]"
5,1085.jpeg,OPEN,OPEN,OPEN,24.3379,16.5911,[0],[0],[0]
6,109.jpeg,OPEN,OPEN,OPEN,25.4539,18.6340,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,24.0617,18.2446,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,27.6473,17.2929,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,27.4751,23.0952,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"


In [43]:
df_night.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,27.6232,18.0765,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,OPEN,30.9799,19.9802,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,28.7311,24.3733,[0],"[0, 0]","[0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,29.6089,17.9242,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,28.2593,30.0285,"[0, 0, 0]","[0, 0, 0, 0]","[0, 1, 0, 0]"
5,1085.jpeg,OPEN,OPEN,OPEN,29.4020,18.1867,[0],[0],[0]
6,109.jpeg,OPEN,OPEN,OPEN,25.3437,18.5799,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,28.2566,17.1756,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,26.6965,20.4973,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,OPEN,33.0346,28.2748,"[0, 0, 0]","[0, 0, 0]","[0, 0, 0]"


In [44]:
df_occlusion.head(10)


,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,CLOSE,CLOSE,25.8429,14.1457,"[0, 0]",[],[]
1,1035.jpeg,OPEN,OPEN,OPEN,32.7612,20.8528,[0],[0],[0]
2,1048.jpeg,OPEN,OPEN,OPEN,43.7688,26.8844,[0],"[0, 0, 0]","[0, 0, 0]"
3,1053.jpeg,OPEN,OPEN,OPEN,26.3380,21.5025,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,OPEN,CLOSE,28.1275,39.0118,"[0, 0, 0]","[0, 0, 0, 0, 0]","[0, 0, 1, 0, 1]"
5,1085.jpeg,OPEN,OPEN,CLOSE,33.1544,32.1422,[0],"[0, 0]","[1, 1]"
6,109.jpeg,OPEN,OPEN,OPEN,27.9312,20.0803,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,29.6079,20.2732,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,31.7391,17.8642,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,CLOSE,29.7518,36.1962,"[0, 0, 0]","[0, 0, 0, 0]","[0, 1, 0, 0]"


In [45]:
df_motion_blur.head(10)

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
0,1000.jpeg,OPEN,OPEN,OPEN,31.9190,19.2632,"[0, 0]",[0],[0]
1,1035.jpeg,OPEN,OPEN,CLOSE,30.1721,19.0809,[0],[0],[1]
2,1048.jpeg,OPEN,CLOSE,CLOSE,32.5006,15.1337,[0],[],[]
3,1053.jpeg,OPEN,OPEN,OPEN,29.3115,18.8177,"[0, 0, 0, 0, 0]",[0],[0]
4,1054.jpeg,OPEN,CLOSE,CLOSE,31.2099,20.1420,"[0, 0, 0]",[1],[1]
5,1085.jpeg,OPEN,OPEN,CLOSE,30.1962,20.3145,[0],[0],[1]
6,109.jpeg,OPEN,OPEN,OPEN,26.6356,30.9592,[0],[0],[0]
7,110.jpeg,OPEN,OPEN,OPEN,27.8230,20.2453,[0],[0],[0]
8,1108.jpeg,OPEN,OPEN,OPEN,32.9949,19.6586,[0],[0],[0]
9,1133.jpeg,OPEN,OPEN,CLOSE,29.2506,20.7769,"[0, 0, 0]",[0],[1]


In [46]:
yolo_results1 = evaluate_system(
    df_dark,
    "yolo",
    "yolo_time_ms"
)
pipeline_results1 = evaluate_system(
    df_dark,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9886
Precision: 0.9897
Recall   : 0.9897
F1-score : 0.9897

TP : 482
FP : 5
TN : 386
FN : 5

Średni czas: 28.74 ms
pipeline
Accuracy : 0.9692
Precision: 0.9582
Recall   : 0.9877
F1-score : 0.9727

TP : 481
FP : 21
TN : 370
FN : 6

Średni czas: 22.44 ms


In [79]:
yolo_results2 = evaluate_system(
    df_night,
    "yolo",
    "yolo_time_ms"
)
pipeline_results2 = evaluate_system(
    df_night,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9727
Precision: 0.9874
Recall   : 0.9630
F1-score : 0.9751

TP : 469
FP : 6
TN : 385
FN : 18

Średni czas: 28.65 ms
pipeline
Accuracy : 0.9294
Precision: 0.9094
Recall   : 0.9692
F1-score : 0.9384

TP : 472
FP : 47
TN : 344
FN : 15

Średni czas: 21.74 ms


In [48]:
yolo_results3 = evaluate_system(
    df_occlusion,
    "yolo",
    "yolo_time_ms"
)
pipeline_results3 = evaluate_system(
    df_occlusion,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.9795
Precision: 0.9916
Recall   : 0.9713
F1-score : 0.9813

TP : 473
FP : 4
TN : 387
FN : 14

Średni czas: 32.08 ms
pipeline
Accuracy : 0.9533
Precision: 0.9785
Recall   : 0.9363
F1-score : 0.9570

TP : 456
FP : 10
TN : 381
FN : 31

Średni czas: 27.47 ms


In [49]:
yolo_results4 = evaluate_system(
    df_motion_blur,
    "yolo",
    "yolo_time_ms"
)
pipeline_results4 = evaluate_system(
    df_motion_blur,
    "pipeline",
    "pipeline_time_ms"
)

yolo
Accuracy : 0.8656
Precision: 0.9301
Recall   : 0.8193
F1-score : 0.8712

TP : 399
FP : 30
TN : 361
FN : 88

Średni czas: 31.04 ms
pipeline
Accuracy : 0.7608
Precision: 0.9826
Recall   : 0.5791
F1-score : 0.7287

TP : 282
FP : 5
TN : 386
FN : 205

Średni czas: 22.85 ms


In [50]:
comparison = pd.DataFrame(
    [
        yolo_results1,
        pipeline_results1
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.988610,0.989733,0.989733,0.989733,482,5,386,5,28.738904
YOLO + CNN,0.969248,0.958167,0.987680,0.972700,481,21,370,6,22.441093


In [51]:
comparison = pd.DataFrame(
    [
        yolo_results2,
        pipeline_results2

    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.972665,0.987368,0.963039,0.975052,469,6,385,18,28.647686
YOLO + CNN,0.929385,0.909441,0.969199,0.938370,472,47,344,15,21.742792


In [52]:
comparison = pd.DataFrame(
    [
        yolo_results3,
        pipeline_results3
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.979499,0.991614,0.971253,0.981328,473,4,387,14,32.077634
YOLO + CNN,0.953303,0.978541,0.936345,0.956978,456,10,381,31,27.468096


In [53]:
comparison = pd.DataFrame(
    [
        yolo_results4,
        pipeline_results4
    ],
    index=[
        "YOLO",
        "YOLO + CNN"
    ]
)
comparison

,Accuracy,Precision,Recall,F1,TP,FP,TN,FN,Time
YOLO,0.865604,0.930070,0.819302,0.871179,399,30,361,88,31.040427
YOLO + CNN,0.760820,0.982578,0.579055,0.728682,282,5,386,205,22.847409


In [54]:
dangerous_yolo1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN")
]

In [55]:
dangerous_yolo2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN")
]

In [56]:
dangerous_yolo3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN")
]

In [57]:
dangerous_yolo4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN")
]

In [73]:
dangerous_pipeline1 = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["pipeline"]=="OPEN")
]

In [74]:
dangerous_pipeline2 = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["pipeline"]=="OPEN")
]

In [75]:
dangerous_pipeline3 = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["pipeline"]=="OPEN")
]

In [76]:
dangerous_pipeline4 = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]

In [ ]:
comparison = pd.DataFrame(

    [   

        yolo_results,

        pipeline_results,

        yolo_results1,

        pipeline_results1,

        yolo_results2,

        pipeline_results2,

        yolo_results3,

        pipeline_results3,

        yolo_results4,

        pipeline_results4

    ],

    index=[

        "YOLO normal",

        "YOLO + CNN normal",

        "YOLO dark",

        "YOLO + CNN dark",

        "YOLO night",

        "YOLO + CNN night",

        "YOLO occlusion",

        "YOLO + CNN occlusion",

        "YOLO motion",

        "YOLO + CNN motion"

    ]

)

comparison

In [77]:
print()
print("Krytyczne błędy, wpuszczenie drapieżnika")
print("----------------")
print("YOLO dark:",len(dangerous_yolo1),"    YOLO night:",len(dangerous_yolo2),"    YOLO occlusion:",len(dangerous_yolo3),"    YOLO occlusion:",len(dangerous_yolo4))
print("YOLO+CNN dark:",len(dangerous_pipeline1),"YOLO+CNN night:",len(dangerous_pipeline2),"YOLO+CNN occlusion:",len(dangerous_pipeline3),"YOLO+CNN occlusion:",len(dangerous_pipeline4))


Krytyczne błędy, wpuszczenie drapieżnika
----------------
YOLO dark: 5     YOLO night: 6     YOLO occlusion: 4     YOLO occlusion: 30
YOLO+CNN dark: 21 YOLO+CNN night: 47 YOLO+CNN occlusion: 10 YOLO+CNN occlusion: 5


In [63]:
improved = df_dark[
    (df_dark["ground_truth"]=="CLOSE") &
    (df_dark["yolo"]=="OPEN") &
    (df_dark["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,36.0947,28.2769,[1],[0],[1]


In [64]:
improved = df_night[
    (df_night["ground_truth"]=="CLOSE") &
    (df_night["yolo"]=="OPEN") &
    (df_night["pipeline"]=="CLOSE")

]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,31.6345,20.0271,[1],[0],[1]


In [65]:
improved = df_occlusion[
    (df_occlusion["ground_truth"]=="CLOSE") &
    (df_occlusion["yolo"]=="OPEN") &
    (df_occlusion["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
300,Image-62-e710b5.jpg,CLOSE,OPEN,CLOSE,31.1292,24.0711,[1],[0],[1]
339,Image-88-8f30f6.jpg,CLOSE,OPEN,CLOSE,26.4056,18.7701,[1],[0],[1]


In [66]:
improved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="OPEN") &
    (df_motion_blur["pipeline"]=="CLOSE")
]
improved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
91,coyote__lila_AMMonitor_Camera_Traps_NEK-VB1_00...,CLOSE,OPEN,CLOSE,28.3639,20.3172,[1],[0],[1]
141,coyote__lila_WSU_Lynx_IMG_0921_jpg.rf.LXZgTLOf...,CLOSE,OPEN,CLOSE,27.9173,20.6003,[1],[0],[1]
166,fox__gbif_fox_0058_jpg.rf.276TrYjTZ8l8fHW2cbDV...,CLOSE,OPEN,CLOSE,29.8053,21.3451,[1],[0],[1]
186,fox__gbif_fox_0454_jpg.rf.ET1jFfTjXFmpusKl9rJT...,CLOSE,OPEN,CLOSE,32.4081,22.5086,[1],[0],[1]
193,fox__gbif_fox_0540_jpg.rf.SviRhT8VBkdRDGFU1QO6...,CLOSE,OPEN,CLOSE,25.0015,19.6134,[1],[0],[1]
196,fox__gbif_fox_0615_jpg.rf.xGBhUyQrn1rpiysNE7WH...,CLOSE,OPEN,CLOSE,29.3695,19.6209,[1],[0],[1]
198,fox__gbif_fox_0722_jpg.rf.4EaY143B8mKPFC0058JJ...,CLOSE,OPEN,CLOSE,28.3034,19.8020,[1],[0],[1]
226,Image-111-ed55ed.jpg,CLOSE,OPEN,CLOSE,31.9443,19.7323,[1],[0],[1]
230,Image-115-0540fa.png,CLOSE,OPEN,CLOSE,35.9259,35.2444,[1],[0],[1]
250,Image-31-1d307b.jpg,CLOSE,OPEN,CLOSE,32.8181,25.3317,[1],[0],[1]


In [72]:
deproved = df_motion_blur[
    (df_motion_blur["ground_truth"]=="CLOSE") &
    (df_motion_blur["yolo"]=="CLOSE") &
    (df_motion_blur["pipeline"]=="OPEN")
]
deproved

,image,ground_truth,yolo,pipeline,yolo_time_ms,pipeline_time_ms,objects_gt,objects_yolo,objects_pipeline
210,fox__lila_WCS_Camera_Traps_0312_jpg.rf.WM0NCxZ...,CLOSE,CLOSE,OPEN,28.0942,17.3235,[1],[1],[0]
874,raptor__raptor_027_jpg.rf.PzIAqRQVSQEonrCHFVSD...,CLOSE,CLOSE,OPEN,29.4113,22.2769,"[1, 1]",[1],[0]


In [67]:
empty_images0 = 0
false_detections0 = 0
for row in results_dark:

    if len(row["objects_gt"]) == 0:

        empty_images0 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections0 += 1

print("Puste obrazy:",empty_images0)
print("Fałszywe detekcje:",false_detections0)
if empty_images0>0:

    print(
        "Odsetek:",
        false_detections0/empty_images0
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [68]:
empty_images1 = 0
false_detections1 = 0

for row in results_night:

    if len(row["objects_gt"]) == 0:

        empty_images1 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections1 += 1

print("Puste obrazy:",empty_images1)
print("Fałszywe detekcje:",false_detections1)

if empty_images1>0:

    print(
        "Odsetek:",
        false_detections1/empty_images1
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [69]:
empty_images2 = 0
false_detections2 = 0

for row in results_occlusion:

    if len(row["objects_gt"]) == 0:

        empty_images2 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections2 += 1

print("Puste obrazy:",empty_images2)
print("Fałszywe detekcje:",false_detections2)

if empty_images2>0:

    print(
        "Odsetek:",
        false_detections2/empty_images2
    )

Puste obrazy: 37
Fałszywe detekcje: 1
Odsetek: 0.02702702702702703


In [70]:
empty_images3 = 0
false_detections3 = 0

for row in results_motion_blur:

    if len(row["objects_gt"]) == 0:

        empty_images3 += 1

        if len(row["objects_yolo"]) > 0:
            false_detections3 += 1

print("Puste obrazy:",empty_images3)
print("Fałszywe detekcje:",false_detections3)

if empty_images3>0:

    print(
        "Odsetek:",
        false_detections3/empty_images3
    )

Puste obrazy: 37
Fałszywe detekcje: 7
Odsetek: 0.1891891891891892


In [71]:

#ZMIEŃ CONFIDENC POTEM NA 0,5 I PORÓWNAJ!!!!!!